# GeeksforGeeks + W3Schools Kaggle Ingestion

This notebook ingests:

- `naidukarthi2193/geeks-for-geeks-articles-dataset`
- `haohoangofficial/w3school-crawling-2025`

It uses the structures observed from the downloaded datasets:

**GeeksforGeeks**
- CSV: `geekstest1.csv`
- Columns: `url`, `title`, `rating`, `content`, `tags`
- The CSV may not be UTF-8, so the loader falls back to `latin-1`.

**W3Schools**
- `metadata.json`
- An `html/` directory containing individual `.html` pages
- HTML pages are parsed with BeautifulSoup instead of being treated as a tabular file.

Both sources are filtered to **AI / Data / Cloud** and standardized to the project schema.


## 1. Imports

In [41]:
from pathlib import Path
import json
import re
import html as html_lib

import pandas as pd
import kagglehub
from bs4 import BeautifulSoup

## 2. Output paths

This notebook assumes it is stored under:

`notebooks/geeks_for_geeks_w3schools/`

Therefore `../../data/raw` points to the repository's shared raw-data folder.


In [42]:
DATA_DIR = Path("../../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

GFG_OUTPUT = DATA_DIR / "geeksforgeeks_articles.json"
W3_OUTPUT = DATA_DIR / "w3schools_content.json"

print("Raw data folder:", DATA_DIR.resolve())

Raw data folder: D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\data\raw


## 3. Download both Kaggle datasets

In [43]:
GFG_DATASET = "naidukarthi2193/geeks-for-geeks-articles-dataset"
W3_DATASET = "haohoangofficial/w3school-crawling-2025"

gfg_path = Path(kagglehub.dataset_download(GFG_DATASET))
w3_path = Path(kagglehub.dataset_download(W3_DATASET))

print("GeeksforGeeks:", gfg_path)
print("W3Schools:", w3_path)

GeeksforGeeks: C:\Users\Sarah\.cache\kagglehub\datasets\naidukarthi2193\geeks-for-geeks-articles-dataset\versions\1
W3Schools: C:\Users\Sarah\.cache\kagglehub\datasets\haohoangofficial\w3school-crawling-2025\versions\3


## 4. Inspect downloaded structure

In [44]:
def summarize_files(folder, max_examples=10):
    files = [p for p in folder.rglob("*") if p.is_file()]
    print(f"Total files: {len(files):,}")

    suffix_counts = {}
    for p in files:
        suffix = p.suffix.lower() or "[no extension]"
        suffix_counts[suffix] = suffix_counts.get(suffix, 0) + 1

    print("File types:", suffix_counts)
    print("Examples:")
    for p in files[:max_examples]:
        print(" -", p.relative_to(folder))

print("=== GeeksforGeeks ===")
summarize_files(gfg_path)

print("\n=== W3Schools ===")
summarize_files(w3_path)

=== GeeksforGeeks ===
Total files: 1
File types: {'.csv': 1}
Examples:
 - geekstest1.csv

=== W3Schools ===
Total files: 36,713
File types: {'.json': 2, '.html': 36711}
Examples:
 - w3school-crawling-2025\metadata.json
 - w3school-crawling-2025\html\0000a7c0-4e50-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\0004d491-4e50-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\00059e60-4cf2-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\0008f516-4cf2-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\000906a9-4e50-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\000d1fc2-4e50-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\000e40c5-4cf2-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\0011366d-4e50-11f0-9280-24b6fdf6f54e.html
 - w3school-crawling-2025\html\0012064b-4cf2-11f0-9280-24b6fdf6f54e.html


# Part A — GeeksforGeeks

## 5. Load the GeeksforGeeks CSV

The downloaded CSV was observed to fail UTF-8 decoding, so this loader first tries UTF-8 and then falls back to `latin-1`.


In [45]:
def load_gfg_csv(folder):
    csv_files = list(folder.rglob("*.csv"))

    if not csv_files:
        raise FileNotFoundError("No CSV file found in the GeeksforGeeks dataset.")

    frames = []

    for file in csv_files:
        try:
            df = pd.read_csv(file, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            print(f"UTF-8 failed for {file.name}, trying latin-1...")
            df = pd.read_csv(file, encoding="latin-1", low_memory=False)

        df["_source_file"] = file.name
        frames.append(df)

        print(f"Loaded {file.name}: {df.shape}")
        print("Columns:", df.columns.tolist())

    return pd.concat(frames, ignore_index=True, sort=False)

gfg_raw = load_gfg_csv(gfg_path)

print("\nGeeksforGeeks total shape:", gfg_raw.shape)
display(gfg_raw.head())

UTF-8 failed for geekstest1.csv, trying latin-1...
Loaded geekstest1.csv: (24489, 6)
Columns: ['url', 'title', 'rating', 'content', 'tags', '_source_file']

GeeksforGeeks total shape: (24489, 6)


,url,title,rating,content,tags,_source_file
0,https://www.geeksforgeeks.org/python-threshold...,Python | Thresholding techniques using OpenCV ...,0.0,Prerequisite: Simple Thresholding using OpenC...,"{'Python', 'OpenCV', 'Image-Processing'}",geekstest1.csv
1,https://www.geeksforgeeks.org/generation-n-num...,Generation of n numbers with given set of factors,0.0,"Given an array of k numbers factor[], the task...","{'prime-factor', 'Mathematical'}",geekstest1.csv
2,https://www.geeksforgeeks.org/c-operators-ques...,C | Operators | Question 17,1.5,Which of the following can have different mean...,"{'C', 'C-Operators', 'Operators', 'C Quiz'}",geekstest1.csv
3,https://www.geeksforgeeks.org/sympy-subset-siz...,SymPy | Subset.size() in Python,0.0,Subset.size() : size() is a sympy Python libra...,{'Python'},geekstest1.csv
4,https://www.geeksforgeeks.org/python-pil-image...,Python PIL | ImageFont.truetype(),0.0,PIL is the Python Imaging Library which provid...,"{'Python', 'Python-pil'}",geekstest1.csv


## 6. Explicit GeeksforGeeks mapping

In [46]:
gfg_mapping = {
    "title": "title",
    "author": None,
    "publication_date": None,
    "url": "url",
    "content": "content",
    "category": None,
    "tags": "tags"
}

print(gfg_mapping)

{'title': 'title', 'author': None, 'publication_date': None, 'url': 'url', 'content': 'content', 'category': None, 'tags': 'tags'}


# Part B — W3Schools

## 7. Locate `metadata.json` and HTML pages


In [47]:
metadata_files = list(w3_path.rglob("metadata.json"))
html_files = list(w3_path.rglob("*.html"))

if not metadata_files:
    raise FileNotFoundError("metadata.json was not found in the W3Schools dataset.")

W3_METADATA_FILE = metadata_files[0]

print("Metadata:", W3_METADATA_FILE)
print(f"HTML pages: {len(html_files):,}")
print("First HTML file:", html_files[0] if html_files else "None")

Metadata: C:\Users\Sarah\.cache\kagglehub\datasets\haohoangofficial\w3school-crawling-2025\versions\3\w3school-crawling-2025\metadata.json
HTML pages: 36,711
First HTML file: C:\Users\Sarah\.cache\kagglehub\datasets\haohoangofficial\w3school-crawling-2025\versions\3\w3school-crawling-2025\html\0000a7c0-4e50-11f0-9280-24b6fdf6f54e.html


## 8. Inspect W3Schools metadata

The dataset stores the actual page bodies as HTML, so metadata is used only when it can supply fields such as URL/title. The parser below does not assume one exact metadata schema.


In [48]:
with open(W3_METADATA_FILE, "r", encoding="utf-8") as f:
    w3_metadata = json.load(f)

print("Metadata Python type:", type(w3_metadata).__name__)

if isinstance(w3_metadata, dict):
    print("Top-level keys:", list(w3_metadata.keys())[:30])
elif isinstance(w3_metadata, list):
    print("Number of metadata entries:", len(w3_metadata))

print("\nMetadata preview:")
print(json.dumps(w3_metadata, ensure_ascii=False, indent=2)[:3000])

Metadata Python type: list
Number of metadata entries: 36290

Metadata preview:
[
  {
    "id": "0000a7c0-4e50-11f0-9280-24b6fdf6f54e",
    "url": "https://www.w3schools.com/cpp/ref_cstring_memcmp.asp",
    "crawled_at": "27/6/2025 07:18:56"
  },
  {
    "id": "0004d491-4e50-11f0-9280-24b6fdf6f54e",
    "url": "https://www.w3schools.com/cpp/ref_cstring_memcpy.asp",
    "crawled_at": "27/6/2025 07:24:17"
  },
  {
    "id": "00059e60-4cf2-11f0-9280-24b6fdf6f54e",
    "url": "https://www.w3schools.com/bootstrap5/tryit.asp?filename=trybs_grid_examples1&stacked=h",
    "crawled_at": "23/6/2025 14:38:11"
  },
  {
    "id": "0008f516-4cf2-11f0-9280-24b6fdf6f54e",
    "url": "https://www.w3schools.com/bootstrap5/tryit.asp?filename=trybs_grid_examples2&stacked=h",
    "crawled_at": "23/6/2025 14:38:11"
  },
  {
    "id": "000906a9-4e50-11f0-9280-24b6fdf6f54e",
    "url": "https://www.w3schools.com/cpp/ref_cstring_memmove.asp",
    "crawled_at": "27/6/2025 07:24:17"
  },
  {
    "id": "000d1fc2-

## 9. Build a flexible metadata lookup

This searches nested metadata objects for dictionaries that appear to describe individual pages. It indexes them by likely file/id/path values so they can be matched to HTML filenames.


In [49]:
def walk_dicts(value):
    if isinstance(value, dict):
        yield value
        for child in value.values():
            yield from walk_dicts(child)
    elif isinstance(value, list):
        for child in value:
            yield from walk_dicts(child)

def first_value(d, keys):
    for key in keys:
        if key in d and d[key] not in (None, ""):
            return d[key]
    return ""

def build_metadata_lookup(metadata):
    lookup = {}

    id_keys = [
        "id", "uuid", "hash", "filename", "file_name", "file",
        "path", "html", "html_file", "local_path"
    ]

    for item in walk_dicts(metadata):
        for key in id_keys:
            value = item.get(key)
            if value in (None, ""):
                continue

            text = str(value)
            candidates = {
                text,
                Path(text).name,
                Path(text).stem
            }

            for candidate in candidates:
                lookup[candidate] = item

    return lookup

w3_metadata_lookup = build_metadata_lookup(w3_metadata)

print(f"Metadata lookup keys: {len(w3_metadata_lookup):,}")
print("Example keys:", list(w3_metadata_lookup.keys())[:10])

Metadata lookup keys: 36,290
Example keys: ['0000a7c0-4e50-11f0-9280-24b6fdf6f54e', '0004d491-4e50-11f0-9280-24b6fdf6f54e', '00059e60-4cf2-11f0-9280-24b6fdf6f54e', '0008f516-4cf2-11f0-9280-24b6fdf6f54e', '000906a9-4e50-11f0-9280-24b6fdf6f54e', '000d1fc2-4e50-11f0-9280-24b6fdf6f54e', '000e40c5-4cf2-11f0-9280-24b6fdf6f54e', '0011366d-4e50-11f0-9280-24b6fdf6f54e', '0012064b-4cf2-11f0-9280-24b6fdf6f54e', '001513bd-4e50-11f0-9280-24b6fdf6f54e']


## 10. HTML extraction helpers

Navigation, scripts, styles, forms, and other page chrome are removed. The parser prefers `<main>` or `<article>` when available and falls back to `<body>`.


In [50]:
def clean_whitespace(text):
    return re.sub(r"\s+", " ", str(text or "")).strip()


def metadata_for_html(file):
    for key in [file.name, file.stem, str(file)]:
        if key in w3_metadata_lookup:
            return w3_metadata_lookup[key]

    return {}


def extract_w3_html(file):
    raw = file.read_text(
        encoding="utf-8",
        errors="replace"
    )

    soup = BeautifulSoup(raw, "html.parser")

    # -------------------------------------------------
    # 1. Remove HTML elements that are not article content
    # -------------------------------------------------

    for tag in soup([
        "script",
        "style",
        "noscript",
        "svg",
        "nav",
        "footer",
        "header",
        "form",
        "button",
        "aside"
    ]):
        tag.decompose()

    # -------------------------------------------------
    # 2. Remove common W3Schools UI / navigation sections
    # -------------------------------------------------

    unwanted_selectors = [
        "#top-nav-bar",
        "#subtopnav",
        "#sidenav",
        "#leftmenuinner",
        "#right",
        "#footer",
        ".topnav",
        ".w3-sidebar",
        ".w3-bar",
        ".w3-hide-small",
        ".ws-black",
        ".ws-grey",
        ".ws-light-green",
        ".user-profile-bottom-wrapper",
        ".signup",
        ".login",
        ".adngin-main_leaderboard",
        ".adngin-sidebar_top"
    ]

    for selector in unwanted_selectors:
        for element in soup.select(selector):
            element.decompose()

    # -------------------------------------------------
    # 3. Get metadata
    # -------------------------------------------------

    meta = metadata_for_html(file)

    # -------------------------------------------------
    # 4. Extract title
    # -------------------------------------------------

    title = ""

    h1 = soup.find("h1")

    if h1:
        title = clean_whitespace(
            h1.get_text(" ", strip=True)
        )

    elif soup.title:
        title = clean_whitespace(
            soup.title.get_text(" ", strip=True)
        )

    metadata_title = first_value(
        meta,
        [
            "title",
            "name",
            "page_title",
            "heading"
        ]
    )

    if not title and metadata_title:
        title = clean_whitespace(metadata_title)

    # -------------------------------------------------
    # 5. Find actual W3Schools tutorial content
    # -------------------------------------------------

    container = (
        soup.select_one("#main")
        or soup.select_one(".w3-main")
        or soup.find("main")
        or soup.find("article")
    )

    # Only fall back to body if no content container exists
    if container is None:
        container = soup.body or soup

    content = clean_whitespace(
        container.get_text(
            " ",
            strip=True
        )
    )

    # -------------------------------------------------
    # 6. Remove remaining common UI phrases
    # -------------------------------------------------

    unwanted_phrases = [
        "Menu",
        "Search field",
        "Sign In",
        "Log in",
        "Get Certified",
        "My Learning",
        "Spaces",
        "Plus",
        "For Teachers"
    ]

    for phrase in unwanted_phrases:
        content = content.replace(phrase, " ")

    content = clean_whitespace(content)

    # -------------------------------------------------
    # 7. URL
    # -------------------------------------------------

    url = first_value(
        meta,
        [
            "url",
            "link",
            "source_url",
            "page_url",
            "href"
        ]
    )

    # -------------------------------------------------
    # 8. Tags
    # -------------------------------------------------

    tags = first_value(
        meta,
        [
            "tags",
            "keywords",
            "category",
            "categories"
        ]
    )

    return {
        "file": file.name,
        "title": title,
        "url": clean_whitespace(url),
        "content": content,
        "tags": tags
    }

## 11. Test W3Schools extraction on one page

Run this before processing every HTML file. It lets you verify that the HTML structure is being read correctly.


In [51]:
if not html_files:
    raise FileNotFoundError("No HTML files were found in the W3Schools dataset.")

w3_example = extract_w3_html(html_files[0])

print("File:", w3_example["file"])
print("Title:", w3_example["title"])
print("URL:", w3_example["url"])
print("Tags:", w3_example["tags"])
print("Content preview:")
print(w3_example["content"][:1000])

File: 0000a7c0-4e50-11f0-9280-24b6fdf6f54e.html
Title: C++ cstring memcmp() function
URL: https://www.w3schools.com/cpp/ref_cstring_memcmp.asp
Tags: 
Content preview:
C++ cstring memcmp() function ❮ CString Functions Example Compare two blocks of memory to see which is greater: char myStr1[] = "ABCD"; char myStr2[] = "ABCE"; int cmp = memcmp(myStr1, myStr2, 4); if(cmp > 0) { cout << myStr1 << " is greater than " << myStr2 << "\n"; } else if(cmp < 0) { cout << myStr2 << " is greater than " << myStr1 << "\n"; } else { cout << myStr1 << " is equal to " << myStr2 << "\n"; } Try it Yourself » Definition and Usage The memcmp() function compares two blocks of memory and returns an integer indicating which one is greater. For this comparison bytes at the same position from both memory blocks are compared one by one starting at position 0 until one of them does not match or the end of the block of memory has been reached. There are three possible scenarios: If the end of the memory blocks is re

# Part C — Topic filtering

## 12. AI / Data / Cloud keywords

The filter uses specific phrases/technologies instead of broad standalone words such as `data`, `cloud`, or `model`, which would create many false positives.


In [52]:
TOPIC_KEYWORDS = {
    "AI": [
        "artificial intelligence", "machine learning", "deep learning",
        "generative ai", "genai", "large language model",
        "large language models", "llm", "llms",
        "natural language processing", "nlp", "computer vision",
        "neural network", "neural networks", "tensorflow",
        "pytorch", "scikit-learn"
    ],
    "Data": [
        "data science", "data scientist", "data engineering",
        "data engineer", "data analytics", "data analysis",
        "big data", "data pipeline", "data pipelines",
        "etl", "elt", "data warehouse", "data warehousing",
        "data lake", "data lakes", "database", "databases",
        "sql", "postgresql", "mysql", "mongodb",
        "spark", "apache spark", "airflow", "apache airflow",
        "dbt", "pandas", "numpy"
    ],
    "Cloud": [
        "cloud computing", "cloud architecture", "cloud infrastructure",
        "cloud engineering", "cloud engineer",
        "aws", "amazon web services", "azure", "microsoft azure",
        "google cloud", "google cloud platform", "gcp",
        "serverless", "kubernetes", "docker", "cloud native"
    ]
}

## 13. Topic classifier

In [53]:
def classify_topics(text):
    text = clean_whitespace(text).lower()
    matched = []

    for topic, keywords in TOPIC_KEYWORDS.items():
        for keyword in keywords:
            # Prevent short tokens such as AI, SQL, AWS, GCP, ETL
            # from matching inside unrelated words.
            if len(keyword) <= 4 and keyword.replace("-", "").isalnum():
                if re.search(rf"(?<!\w){re.escape(keyword)}(?!\w)", text):
                    matched.append(topic)
                    break
            elif keyword in text:
                matched.append(topic)
                break

    return matched

## 14. Filter GeeksforGeeks

Searches title + tags + the first part of the article content.


In [54]:
def safe_text(value):
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except (TypeError, ValueError):
        pass
    return str(value)

def gfg_search_text(row):
    return " ".join([
        safe_text(row.get("title", "")),
        safe_text(row.get("tags", "")),
        safe_text(row.get("content", ""))[:5000]
    ])

gfg_filtered = gfg_raw.copy()
gfg_filtered["_topics"] = gfg_filtered.apply(
    lambda row: classify_topics(gfg_search_text(row)),
    axis=1
)
gfg_filtered = gfg_filtered[
    gfg_filtered["_topics"].map(len) > 0
].copy()

print(f"GeeksforGeeks: {len(gfg_raw):,} raw -> {len(gfg_filtered):,} relevant")

GeeksforGeeks: 24,489 raw -> 2,466 relevant


## 15. Parse and filter W3Schools incrementally

The W3Schools download contains many HTML files, so pages are parsed one at a time. Only AI/Data/Cloud matches are kept in memory.


In [55]:
# Limit how many W3Schools pages we process
MAX_HTML_FILES = 2000

files_to_process = html_files[:MAX_HTML_FILES]

print(
    f"Processing {len(files_to_process):,} out of "
    f"{len(html_files):,} available HTML pages..."
)

w3_relevant_records = []

for i, file in enumerate(files_to_process, start=1):

    # Extract the page first
    record = extract_w3_html(file)

    # ---------------------------------------------
    # CLASSIFICATION TEXT
    # Only use title + tags.
    # Do NOT use the entire page content.
    # ---------------------------------------------
    classification_text = " ".join([
        safe_text(record["title"]),
        safe_text(record["tags"])
    ])

    topics = classify_topics(classification_text)

    # Only keep AI / Data / Cloud pages
    if topics:
        record["_topics"] = topics
        w3_relevant_records.append(record)

    # Progress
    if i % 500 == 0 or i == len(files_to_process):
        print(
            f"Processed {i:,}/{len(files_to_process):,} HTML pages | "
            f"relevant: {len(w3_relevant_records):,}"
        )


w3_filtered = pd.DataFrame(w3_relevant_records)

print(
    f"\nW3Schools: {len(files_to_process):,} HTML pages processed -> "
    f"{len(w3_filtered):,} relevant"
)

display(w3_filtered.head())

Processing 2,000 out of 36,711 available HTML pages...
Processed 500/2,000 HTML pages | relevant: 34
Processed 1,000/2,000 HTML pages | relevant: 61
Processed 1,500/2,000 HTML pages | relevant: 74
Processed 2,000/2,000 HTML pages | relevant: 97

W3Schools: 2,000 HTML pages processed -> 97 relevant


,file,title,url,content,tags,_topics
0,0016ec42-4d21-11f0-9280-24b6fdf6f54e.html,MySQL Tryit Editor v1.0,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,,[Data]
1,00697f75-4d2c-11f0-9280-24b6fdf6f54e.html,MySQL Tryit Editor v1.0,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,,[Data]
2,006d5e19-4d2c-11f0-9280-24b6fdf6f54e.html,MySQL Tryit Editor v1.0,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,,[Data]
3,0071250e-4d2c-11f0-9280-24b6fdf6f54e.html,MySQL Tryit Editor v1.0,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,,[Data]
4,0071aab0-4d31-11f0-9280-24b6fdf6f54e.html,MySQL Tryit Editor v1.0,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,,[Data]


# Part D — Standardization

## 16. Standardize GeeksforGeeks

The source does not provide author or publication date, so those fields are left blank rather than invented.


In [56]:
gfg_standardized = pd.DataFrame({
    "source": ["GeeksforGeeks"] * len(gfg_filtered),
    "category": gfg_filtered["_topics"].apply(lambda x: ", ".join(x)),
    "title": gfg_filtered["title"].fillna("").astype(str),
    "author": [""] * len(gfg_filtered),
    "publication_date": [""] * len(gfg_filtered),
    "description": [""] * len(gfg_filtered),
    "url": gfg_filtered["url"].fillna("").astype(str),
    "content": gfg_filtered["content"].fillna("").astype(str),
    "tags": gfg_filtered["tags"].fillna("").astype(str)
}).reset_index(drop=True)

display(gfg_standardized.head())

,source,category,title,author,publication_date,description,url,content,tags
0,GeeksforGeeks,Data,Python | Pandas Series.transform(),,,,https://www.geeksforgeeks.org/python-pandas-se...,Pandas series is a One-dimensional ndarray wit...,"{'Python', 'Python-pandas', 'Python pandas-ser..."
1,GeeksforGeeks,Data,Python | Numpy MaskedArray.__rpow__,,,,https://www.geeksforgeeks.org/python-numpy-mas...,numpy.ma.MaskedArray class is a subclass of nd...,"{'Python', 'Python numpy-ndarray', 'Python-num..."
2,GeeksforGeeks,Data,numpy.not_equal() in Python,,,,https://www.geeksforgeeks.org/numpy-not_equal-...,"About : \nnumpy.not_equal(x1, x2[, out]) : che...","{'Python', 'Python-numpy', 'Python numpy-Logic..."
3,GeeksforGeeks,Data,Python program to count words in a sentence,,,,https://www.geeksforgeeks.org/python-program-t...,Data preprocessing is an important task in tex...,"{'Python', 'Python Programs', 'Python string-p..."
4,GeeksforGeeks,AI,Text Preprocessing in Python | Set ? 1,,,,https://www.geeksforgeeks.org/text-preprocessi...,Prerequisites: Introduction to NLPWhenever we ...,"{'Machine Learning', 'Natural-language-process..."


## 17. Standardize W3Schools

The downloaded W3Schools structure gives us HTML page content and metadata. Fields not available in the source are left blank.


In [57]:
if w3_filtered.empty:
    w3_standardized = pd.DataFrame(columns=[
        "source", "category", "title", "author",
        "publication_date", "description", "url",
        "content", "tags"
    ])
else:
    w3_standardized = pd.DataFrame({
        "source": ["W3Schools"] * len(w3_filtered),
        "category": w3_filtered["_topics"].apply(lambda x: ", ".join(x)),
        "title": w3_filtered["title"].fillna("").astype(str),
        "author": [""] * len(w3_filtered),
        "publication_date": [""] * len(w3_filtered),
        "description": [""] * len(w3_filtered),
        "url": w3_filtered["url"].fillna("").astype(str),
        "content": w3_filtered["content"].fillna("").astype(str),
        "tags": w3_filtered["tags"].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, (list, dict))
            else safe_text(x)
        )
    }).reset_index(drop=True)

display(w3_standardized.head())

,source,category,title,author,publication_date,description,url,content,tags
0,W3Schools,Data,MySQL Tryit Editor v1.0,,,,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,
1,W3Schools,Data,MySQL Tryit Editor v1.0,,,,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,
2,W3Schools,Data,MySQL Tryit Editor v1.0,,,,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,
3,W3Schools,Data,MySQL Tryit Editor v1.0,,,,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,
4,W3Schools,Data,MySQL Tryit Editor v1.0,,,,https://www.w3schools.com/sql/trymysql.asp?fil...,The Try-MySQL Editor at w3schools.com This SQL...,


## 18. Basic data-quality cleaning

In [58]:
def clean_standardized(df):
    result = df.copy()

    for col in result.columns:
        if result[col].dtype == "object":
            result[col] = result[col].apply(
                lambda x: x.strip() if isinstance(x, str) else x
            )

    # A useful record must contain a title or content.
    result = result[
        (result["title"] != "") | (result["content"] != "")
    ].copy()

    # Deduplicate URL-bearing records by URL.
    with_url = result[result["url"] != ""].drop_duplicates(subset=["url"])

    # If URL is unavailable, deduplicate by source/title/content.
    without_url = result[result["url"] == ""].drop_duplicates(
        subset=["source", "title", "content"]
    )

    return pd.concat(
        [with_url, without_url],
        ignore_index=True
    )

gfg_final = clean_standardized(gfg_standardized)
w3_final = clean_standardized(w3_standardized)

print("GeeksforGeeks final:", gfg_final.shape)
print("W3Schools final:", w3_final.shape)

GeeksforGeeks final: (2454, 9)
W3Schools final: (97, 9)


## 19. Topic distributions

In [59]:
print("=== GeeksforGeeks ===")
print(gfg_final["category"].str.split(", ").explode().value_counts())

print("\n=== W3Schools ===")
print(w3_final["category"].str.split(", ").explode().value_counts())

=== GeeksforGeeks ===
category
Data     2004
AI        548
Cloud      11
Name: count, dtype: int64

=== W3Schools ===
category
Data     95
Cloud     2
Name: count, dtype: int64


## 20. Manually validate random samples

Keyword filtering should always be spot-checked before treating the output as curated data.


In [60]:
def show_sample(df, name, n=10):
    print(f"\n===== {name} =====")

    if df.empty:
        print("No matching records.")
        return

    sample = df.sample(min(n, len(df)), random_state=42)

    for _, row in sample.iterrows():
        print("-" * 80)
        print("TITLE:", row["title"])
        print("CATEGORY:", row["category"])
        print("URL:", row["url"])
        print("CONTENT:", row["content"][:250])

show_sample(gfg_final, "GeeksforGeeks")
show_sample(w3_final, "W3Schools")


===== GeeksforGeeks =====
--------------------------------------------------------------------------------
TITLE: numpy string operations | swapcase() function
CATEGORY: Data
URL: https://www.geeksforgeeks.org/numpy-string-operations-swapcase-function/
CONTENT: numpy.core.defchararray.swapcase(arr) function return element-wise a copy of the string with uppercase characters converted to lowercase and lowercase characters converted to uppercase.
Syntax:  numpy.char.swapcase(arr)Parameters:
arr  : [ array_like
--------------------------------------------------------------------------------
TITLE: Python | Numpy matrix.squeeze()
CATEGORY: Data
URL: https://www.geeksforgeeks.org/python-numpy-matrix-squeeze/
CONTENT: With the help of matrix.squeeze() method, we are able to squeeze the size of a matrix by using the same method. But remember one thing we use this method on Nx1 size of matrix which gives out as 1xN matrix.
Syntax :   matrix.squeeze()
Return :  Retur
---------------------------

## 21. Save JSON outputs to `data/raw/`

In [61]:
def save_json(df, path):
    records = df.to_dict(orient="records")

    with open(path, "w", encoding="utf-8") as f:
        json.dump(
            records,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Saved {len(records):,} records -> {path.resolve()}")

save_json(gfg_final, GFG_OUTPUT)
save_json(w3_final, W3_OUTPUT)

Saved 2,454 records -> D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\data\raw\geeksforgeeks_articles.json
Saved 97 records -> D:\_SDA-WCD-DataEngineeringBootcamp\TechHub-Group3\data\raw\w3schools_content.json


## 22. Verify saved files

In [62]:
for path in [GFG_OUTPUT, W3_OUTPUT]:
    with open(path, "r", encoding="utf-8") as f:
        records = json.load(f)

    print(path.name, ":", len(records), "records")

    if records:
        print(
            json.dumps(
                records[0],
                ensure_ascii=False,
                indent=2
            )[:2000]
        )

    print("=" * 80)

geeksforgeeks_articles.json : 2454 records
{
  "source": "GeeksforGeeks",
  "category": "Data",
  "title": "Python | Pandas Series.transform()",
  "author": "",
  "publication_date": "",
  "description": "",
  "url": "https://www.geeksforgeeks.org/python-pandas-series-transform/",
  "content": "Pandas series is a One-dimensional ndarray with axis labels. The labels need not be unique but must be a hashable type. The object supports both integer- and label-based indexing and provides a host of methods for performing operations involving the index.Pandas Series.transform() function Call func (the passed function) on self producing a Series with transformed values and that has the same axis length as self.Syntax: Series.transform(func, axis=0, *args, **kwargs)Parameter : \nfunc :  If a function, must either work when passed a Series or when passed to Series.apply\naxis :  Parameter needed for compatibility with DataFrame.\n*args :  Positional arguments to pass to func.\n**kwargs :   Keywo

## Output

The notebook produces:

- `data/raw/geeksforgeeks_articles.json`
- `data/raw/w3schools_content.json`

These remain raw-source ingestion outputs. Your later profiling, cleaning, validation, and join/transform notebooks can combine them with Medium, Pluralsight, and the team's other sources.
